In [0]:
# 1. Instalação e Importações
%pip install yfinance

import yfinance as yf
from pyspark.sql.functions import col, lit, round
import pandas as pd
from datetime import datetime

# 2. Configurações
tickers = ["PETR4.SA", "VALE3.SA", "ITUB4.SA", "ABEV3.SA", "BBDC4.SA", "WEGE3.SA", "BBAS3.SA"]
table_name = "default.monitoramento_acoes_br_long"
data_inicio = "2022-01-01"
hoje = datetime.now().strftime('%Y-%m-%d')

# 3. Extração e Consolidação
full_spark_df = None

for ticker in tickers:
    # Download e limpeza inicial do Pandas
    pdf = yf.download(ticker, start=data_inicio, end=hoje, auto_adjust=True).reset_index()
    
    if isinstance(pdf.columns, pd.MultiIndex):
        pdf.columns = [c[0] if isinstance(c, tuple) else c for c in pdf.columns]
    
    # Ajuste de tipos para compatibilidade Spark
    pdf['Date'] = pd.to_datetime(pdf['Date'])
    pdf['Close'] = pd.to_numeric(pdf['Close'], errors='coerce').astype(float)
    
    # Conversão para Spark e união
    temp_df = spark.createDataFrame(pdf[['Date', 'Close']]).withColumn("Ticker", lit(ticker.replace(".SA", "")))
    full_spark_df = temp_df if full_spark_df is None else full_spark_df.union(temp_df)

# 4. Transformação Final e Carga
df_final = (full_spark_df
    .withColumn("preco", round(col("Close"), 2))
    .withColumn("data", col("Date").cast("date"))
    .select("data", "Ticker", "preco")
)

df_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

# 5. Visualização
display(spark.table(table_name).orderBy(col("data"), ascending=False))

[*********************100%***********************]  1 of 1 completed

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


data,Ticker,preco
2026-04-16,ITUB4,46.98
2026-04-16,ABEV3,15.43
2026-04-16,PETR4,48.58
2026-04-16,VALE3,87.44
2026-04-16,BBDC4,20.85
2026-04-16,BBAS3,24.28
2026-04-16,WEGE3,48.4
2026-04-15,ITUB4,47.04
2026-04-15,VALE3,88.44
2026-04-15,BBDC4,20.8
